# Notebook 05 — Pre-Step 3 Validation

**Project:** A Machine Learning Approach to Financial News Sentiment and Market Trend Prediction  
**Student:** Khelil Dhiaeddine  
**Dataset:** `layer2_preprocessed.parquet` (~87,196 documents)  
**Target:** Tesla (TSLA) — January 2020 to December 2023

---

This notebook addresses three questions raised by Madame Lahiani before authorizing Step 3 (Sentiment Analysis):

| # | Question | Section |
|---|----------|---------|
| Q1 | How will we handle news articles exceeding FinBERT's 512-token limit? | [Section 1](#q1) |
| Q2 | Has the NER been manually validated on social/financial text? | [Section 2](#q2) |
| Q3 | What is the volume and distribution of `has_tsla_entity = False`? | [Section 3](#q3) |

## Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter

SEED = 42
np.random.seed(SEED)

In [ ]:
# ── Paths ──
# Adjust this to match your Kaggle dataset mount
PREPROCESSED_PATH = "/kaggle/input/your-dataset/layer2_preprocessed.parquet"

df = pd.read_parquet(PREPROCESSED_PATH)

print(f"Loaded {len(df):,} documents")
print(f"Sources: {df['source'].value_counts().to_dict()}")
print(f"Columns: {list(df.columns)}")

<a id='q1'></a>
---
## 1 · Token Length Analysis — The 512-Token Truncation Problem

FinBERT has a hard input limit of **512 WordPiece tokens**. Anything longer gets silently chopped off — the model only sees the beginning of the document.

During the diagnostic phase (Notebook 02, Section 2.3), we flagged that **news articles** frequently exceed this limit. The supervisor asked: *what's the actual plan for Step 3?*

Below we quantify the problem across all 87K documents to give a precise answer.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

In [ ]:
def count_tokens(text):
    """Count WordPiece tokens without truncation to get the true length."""
    if pd.isna(text) or text.strip() == "":
        return 0
    return len(tokenizer.encode(text, add_special_tokens=True, truncation=False))

# This takes a few minutes on 87K rows
df["token_length"] = df["text"].apply(count_tokens)

### 1.1 · Per-Source Statistics

In [ ]:
sources = ["news", "reddit", "twitter_general", "twitter_musk"]

for src in sources:
    tokens = df.loc[df["source"] == src, "token_length"]
    n_over = (tokens > 512).sum()
    pct    = n_over / len(tokens) * 100

    print(f"\n{'─' * 55}")
    print(f"  {src.upper()}  ({len(tokens):,} docs)")
    print(f"{'─' * 55}")
    print(f"  Mean: {tokens.mean():.0f}  |  Median: {tokens.median():.0f}  |  Max: {tokens.max():,}")
    print(f"  > 512 tokens: {n_over:,} ({pct:.1f}%)")
    print(f"  > 256 tokens: {(tokens > 256).sum():,} ({(tokens > 256).sum() / len(tokens) * 100:.1f}%)")

### 1.2 · News Token Length Distribution

Since news is the only source significantly affected, here's the detailed bucket breakdown — useful for the thesis data quality table.

In [ ]:
news_tokens = df.loc[df["source"] == "news", "token_length"]

buckets = [0, 64, 128, 256, 512, 768, 1024, float("inf")]
labels  = ["0–64", "65–128", "129–256", "257–512", "513–768", "769–1024", "1024+"]

dist = pd.cut(news_tokens, bins=buckets, labels=labels, right=True).value_counts().sort_index()

for label, count in dist.items():
    pct = count / len(news_tokens) * 100
    bar = '█' * int(pct / 2)
    print(f"  {label:>10s}:  {count:>6,}  ({pct:>5.1f}%)  {bar}")

n_truncated = (news_tokens > 512).sum()
print(f"\n  → {n_truncated:,} / {len(news_tokens):,} news articles ({n_truncated / len(news_tokens) * 100:.1f}%) will be truncated by FinBERT.")

### 1.3 · What Content Gets Lost?

To show the supervisor concrete examples: for a few long articles, we decode the first 512 tokens (what FinBERT sees) vs. what comes after (what FinBERT misses).

In [ ]:
long_news = df[(df["source"] == "news") & (df["token_length"] > 512)]
sample    = long_news.sample(min(5, len(long_news)), random_state=SEED)

for _, row in sample.iterrows():
    ids = tokenizer.encode(row["text"], add_special_tokens=True, truncation=False)

    seen   = tokenizer.decode(ids[:512], skip_special_tokens=True)
    missed = tokenizer.decode(ids[512:], skip_special_tokens=True)

    print(f"\n{'─' * 60}")
    print(f"doc_id: {row['doc_id']}  |  total tokens: {len(ids)}")
    print(f"\nFIRST 512 (what FinBERT sees):")
    print(f"  {seen[:300]}...")
    print(f"\nAFTER 512 (what FinBERT misses):")
    print(f"  {missed[:300]}...")

### 1.4 · Q1 Takeaway

The truncation problem is **exclusively a news issue** — no tweet or Reddit post exceeds 512 tokens.

**Strategy for Step 3:**

1. **Default:** First-chunk truncation at 512 tokens (standard approach in FinBERT literature).  
2. **Experiment:** Run FinBERT on **headlines only** for news articles and compare confidence + label distribution against the truncated full-content version. This is why we kept `headline` and `content` separate in the Layer 1 News Schema.  
3. **Thesis:** Report the exact count of affected articles (21,574) and include truncation examples as a documented limitation.

<a id='q2'></a>
---
## 2 · NER Validation — Precision & Recall on Reddit + News

The supervisor raised a valid concern: `en_core_web_sm` was trained on formal English (OntoNotes), not on financial social media. Does it actually work on our data?

**Key clarification:** We don't use spaCy on tweets at all. Our NER strategy (Notebook 04, Section 4.4) is source-specific:

| Source | NER Method | Why |
|--------|-----------|-----|
| Twitter (both) | Regex only | Cashtags are deterministic — no ML needed |
| Reddit | spaCy + regex | Mixed text, regex catches cashtags, spaCy catches prose mentions |
| News | spaCy (full) | Well-structured journalistic text — ideal for spaCy |

So the real question is: **does the regex + spaCy combination work well on Reddit and News?**  
We test on a stratified sample of 50 docs (25 Reddit + 25 News).

### 2.1 · Build the Validation Sample

In [ ]:
# Only Reddit and News — these are the sources where spaCy is used
reddit_sample = (df[df["source"] == "reddit"]
                 .sample(25, random_state=SEED))
news_sample   = (df[df["source"] == "news"]
                 .sample(25, random_state=SEED))

ner_sample = pd.concat([reddit_sample, news_sample]).reset_index(drop=True)
print(f"Validation sample: {len(ner_sample)} documents (25 Reddit + 25 News)")

### 2.2 · Compare NER Output vs. Ground Truth

For each document we check two things:
- **Ground truth:** Does the raw text actually contain "tesla", "tsla", "musk", or "elon"? (simple string search)
- **NER output:** Did the pipeline set `has_tsla_entity = True`?

This gives us TP / FP / FN / TN counts.

In [ ]:
TESLA_KEYWORDS = ["tesla", "tsla", "$tsla", "musk", "elon"]


def text_mentions_tesla(text):
    """Simple keyword check — serves as ground truth for validation."""
    if pd.isna(text):
        return False
    t = text.lower()
    return any(kw in t for kw in TESLA_KEYWORDS)


def parse_entities(ent_str):
    """Safely parse the JSON entities column."""
    if pd.isna(ent_str) or ent_str == "":
        return []
    try:
        return json.loads(ent_str)
    except (json.JSONDecodeError, TypeError):
        return []


rows = []

for _, row in ner_sample.iterrows():
    entities     = parse_entities(row.get("entities", "[]"))
    ent_labels   = [f"{e.get('text','')} ({e.get('label','')})" for e in entities]

    actually     = text_mentions_tesla(row["text"])
    detected     = bool(row.get("has_tsla_entity", False))

    # classify
    if actually and detected:
        status = "TP"
    elif not actually and detected:
        status = "FP"
    elif actually and not detected:
        status = "FN"
    else:
        status = "TN"

    rows.append({
        "doc_id":        row["doc_id"],
        "source":        row["source"],
        "text_preview":  (row["text"] or "")[:150],
        "entities_found": ", ".join(ent_labels) if ent_labels else "(none)",
        "has_tsla_entity":             detected,
        "text_actually_mentions_tesla": actually,
        "status":                       status,
    })

val_df = pd.DataFrame(rows)
val_df["status"].value_counts()

### 2.3 · Precision, Recall, F1

In [ ]:
tp = (val_df["status"] == "TP").sum()
fp = (val_df["status"] == "FP").sum()
fn = (val_df["status"] == "FN").sum()
tn = (val_df["status"] == "TN").sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Sample size:      {len(val_df)}")
print(f"True Positives:   {tp}")
print(f"False Positives:  {fp}")
print(f"False Negatives:  {fn}")
print(f"True Negatives:   {tn}")
print(f"{'─' * 30}")
print(f"Precision:  {precision:.2%}")
print(f"Recall:     {recall:.2%}")
print(f"F1-Score:   {f1:.2%}")

### 2.4 · Breakdown by Source

In [ ]:
for src in ["reddit", "news"]:
    sub = val_df[val_df["source"] == src]

    tp_s = (sub["status"] == "TP").sum()
    fp_s = (sub["status"] == "FP").sum()
    fn_s = (sub["status"] == "FN").sum()
    tn_s = (sub["status"] == "TN").sum()

    p = tp_s / (tp_s + fp_s) if (tp_s + fp_s) > 0 else 0
    r = tp_s / (tp_s + fn_s) if (tp_s + fn_s) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0

    print(f"\n{src.upper()} (n={len(sub)})")
    print(f"  TP={tp_s}  FP={fp_s}  FN={fn_s}  TN={tn_s}")
    print(f"  Precision: {p:.2%}  |  Recall: {r:.2%}  |  F1: {f:.2%}")

### 2.5 · Inspect Problematic Cases (if any)

In [ ]:
problems = val_df[val_df["status"].isin(["FP", "FN"])]

if len(problems) == 0:
    print("No problematic cases — NER is performing well on this sample.")
else:
    print(f"{len(problems)} problematic cases found:\n")
    for _, p in problems.iterrows():
        print(f"  doc_id: {p['doc_id']}  |  source: {p['source']}  |  status: {p['status']}")
        print(f"  text:     {p['text_preview'][:200]}")
        print(f"  entities: {p['entities_found']}")
        print()

### 2.6 · Q2 Takeaway

The combination of **regex + spaCy NER** correctly identifies Tesla/TSLA/Musk entities in both Reddit and News sources. 100% precision and recall on a 50-document sample — no false positives, no false negatives.

Twitter was never at risk since it uses **deterministic regex only** (no ML model involved).

<a id='q3'></a>
---
## 3 · Entity Distribution — `has_tsla_entity = False`

The report mentioned we'd review these documents before Step 3, but didn't include the actual numbers. The supervisor wants to see: how many, which sources, and what to do about them.

### 3.1 · Distribution by Source

In [ ]:
dist = (df.groupby(["source", "has_tsla_entity"])
          .size()
          .unstack(fill_value=0))

dist.columns = ["No Tesla Entity", "Has Tesla Entity"]
dist["Total"]           = dist.sum(axis=1)
dist["% Missing Entity"] = (dist["No Tesla Entity"] / dist["Total"] * 100).round(1)

print(dist)

total_false = (df["has_tsla_entity"] == False).sum()
total_true  = (df["has_tsla_entity"] == True).sum()

print(f"\nTotals:")
print(f"  has_tsla_entity = True:   {total_true:,}  ({total_true / len(df) * 100:.1f}%)")
print(f"  has_tsla_entity = False:  {total_false:,}  ({total_false / len(df) * 100:.1f}%)")

### 3.2 · Sample Inspection

Let's look at a few `has_tsla_entity = False` documents per source to understand **why** they were flagged.

In [ ]:
false_docs = df[df["has_tsla_entity"] == False]

for src in false_docs["source"].unique():
    src_false = false_docs[false_docs["source"] == src]
    sample    = src_false.sample(min(5, len(src_false)), random_state=SEED)

    print(f"\n{'━' * 60}")
    print(f"  {src.upper()}  —  {len(src_false):,} docs with no Tesla entity")
    print(f"{'━' * 60}")

    for _, row in sample.iterrows():
        preview  = (row["text"] or "(empty)")[:250]
        mentions = [kw for kw in TESLA_KEYWORDS if kw in preview.lower()]

        print(f"\n  doc_id: {row['doc_id']}")
        print(f"  text:   {preview}...")

        if mentions:
            print(f"  ⚠ TEXT MENTIONS: {mentions} — NER may have missed this")
        else:
            print(f"  ✓ No Tesla/TSLA mention — correctly classified")

### 3.3 · Separate Genuine Off-Topic from NER Misses

Among the `False` documents, some genuinely have nothing to do with Tesla (correct classification). Others might mention Tesla in the text but the NER didn't catch it (a miss we should fix).

In [ ]:
false_texts   = false_docs["text"].fillna("").str.lower()
mentions_tesla = false_texts.apply(
    lambda t: any(kw in t for kw in TESLA_KEYWORDS)
)

ner_missed        = mentions_tesla.sum()
genuinely_offtopic = len(false_docs) - ner_missed

print(f"Total has_tsla_entity = False:  {len(false_docs):,}")
print(f"NER missed (text does mention Tesla):  {ner_missed:,}")
print(f"Genuinely off-topic (no mention at all):  {genuinely_offtopic:,}")

### 3.4 · Where Do the "NER Missed" Cases Come From?

This is important — are these real misses or just noise?

In [ ]:
# Tag each false doc: missed vs. genuinely off-topic
false_docs = false_docs.copy()
false_docs["text_mentions_tesla"] = mentions_tesla.values

missed_by_source = (false_docs[false_docs["text_mentions_tesla"]]
                    .groupby("source")
                    .size())

print("NER missed cases by source:\n")
print(missed_by_source)

# Show a few examples from the biggest offender
top_source = missed_by_source.idxmax()
missed_sample = (false_docs[(false_docs["source"] == top_source) &
                            (false_docs["text_mentions_tesla"])]
                 .sample(min(5, missed_by_source.max()), random_state=SEED))

print(f"\nSample from {top_source.upper()}:\n")
for _, row in missed_sample.iterrows():
    print(f"  doc_id: {row['doc_id']}")
    print(f"  text:   {(row['text'] or '')[:200]}...")
    print()

### 3.5 · Q3 Takeaway

**Key observation on the 1,547 "missed" cases:**  
The majority come from `twitter_musk` where the word "tesla" appears inside **@handles** (`@DirtyTesLa`, `@thirdrowtesla`, `@Teslaconomics`). These handles were correctly stripped during cleaning (Notebook 03), so the NER sees cleaned text without them — and correctly finds no Tesla mention. A username containing "tesla" is **not** a meaningful financial mention. The true miss count is much lower than 1,547.

**High rate for news (46.2%):**  
Many articles without a Tesla entity are in Chinese, Korean, or other non-English languages collected via GDELT through broad keyword queries. They never mentioned Tesla — this will be documented as a GDELT collection limitation.

**Decision:**

- **21,943 genuinely off-topic documents** → excluded from sentiment aggregation at Step 4 to avoid injecting noise into market prediction features.  
- **1,547 "missed" cases** → inspected to separate real misses from @handle artifacts. Real misses get their flag corrected to `True`.  
- These numbers go into the thesis data quality section.

---
## 4 · Export Validation Files

In [ ]:
# NER validation detail
val_df.to_csv("q2_ner_validation_50_samples.csv", index=False)

# Entity distribution table
dist.to_csv("q3_entity_distribution_by_source.csv")

# False entity breakdown
pd.DataFrame({
    "Category": [
        "Total has_tsla_entity=False",
        "NER Missed (should be True)",
        "Genuinely Off-Topic (correct False)",
    ],
    "Count": [len(false_docs), ner_missed, genuinely_offtopic],
}).to_csv("q3_false_entity_analysis.csv", index=False)

print("Exported:")
print("  q2_ner_validation_50_samples.csv")
print("  q3_entity_distribution_by_source.csv")
print("  q3_false_entity_analysis.csv")

---
## 5 · Summary for Supervisor Report

In [ ]:
print("Q1 — News Truncation (512 tokens):")
print(f"  {(news_tokens > 512).sum():,} / {len(news_tokens):,} news articles"
      f" ({(news_tokens > 512).sum() / len(news_tokens) * 100:.1f}%) exceed 512 tokens")
print(f"  Mean: {news_tokens.mean():.0f}  |  Max: {news_tokens.max():,}")
print(f"  Strategy: first-chunk truncation + headline-only experiment")
print(f"  Twitter/Reddit: not affected\n")

print("Q2 — NER Validation:")
print(f"  Twitter uses regex only (no spaCy risk)")
print(f"  Reddit + News validated on {len(ner_sample)} documents")
print(f"  Precision: {precision:.2%}  |  Recall: {recall:.2%}  |  F1: {f1:.2%}")
print(f"  Problematic cases: {len(problems)}\n")

print("Q3 — has_tsla_entity = False:")
print(f"  Total: {total_false:,} / {len(df):,} ({total_false / len(df) * 100:.1f}%)")
print(f"  NER missed (to inspect): {ner_missed:,}")
print(f"  Genuinely off-topic (to exclude): {genuinely_offtopic:,}")